# Endometrial Cancer (EC) PRS Extraction and Scoring — All of Us CDR v9

Extracts the EC PRS SNP weights (O'Mara/Wang, supplied by Tracy O'Mara's group, lifted to GRCh38) from the AoU WGS `acaf_threshold` callset, resolves allele orientation against the extracted REF/ALT alleles, and scores each participant with PLINK2 to produce `prs_scores_final_complete.csv` for the final cohort from `EC_Cohort_Definition.ipynb` (2,386 cases / 269,416 controls, N = 271,802).


## Setup


In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np

project_id = os.environ.get('GOOGLE_PROJECT', '').strip().strip("'\"")
DSUB_USER_NAME = os.environ.get('OWNER_EMAIL', '').strip().strip("'\"").split('@')[0].replace('.', '-')
%run /home/jupyter/Var_set_2.ipynb
print(project_id, DSUB_USER_NAME)

In [ ]:
# Confirming dsub is a recent-enough version (0.5.5+ fixes a platform blocker)
result = subprocess.run(['dsub', '--version'], capture_output=True, text=True)
print(result.stdout)

## Variant extraction


*A note on `prs_weights_hg38.csv`: the supplied file (`ukbb_ec_ecac2018_sbayesr_band.snpRes`, from Tracy O'Mara's group) is confirmed GRCh37 (its SNP positions match known GRCh37 coordinates), so the GRCh37 -> GRCh38 liftover below was performed to produce it. Kept commented out since the output file already exists in our AoU workbench; uncomment to regenerate it. Success rate: 99.961% (1,138,676 / 1,139,118 SNPs lifted).*

In [ ]:
# GRCh37->GRCh38 liftover of the EC weights
# Kept commented out; not needed to reproduce prs_weights_hg38.csv, which already exists. 
# Uncomment to regenerate it from scratch.

# from pyliftover import LiftOver
# lo = LiftOver('hg19', 'hg38')
# ec_weights_raw = pd.read_csv('/home/jupyter/ukbb_ec_ecac2018_sbayesr_band.snpRes', sep=r'\s+')
#
# def lift_pos(row):
#     result = lo.convert_coordinate(f"chr{row['Chrom']}", int(row['Position']) - 1)
#     if result and len(result) > 0:
#         return result[0][1] + 1
#     return None
#
# ec_weights_raw['position_hg38'] = ec_weights_raw.apply(lift_pos, axis=1)
# n_lifted = ec_weights_raw['position_hg38'].notna().sum()
# print(f"Lifted: {n_lifted:,} / {len(ec_weights_raw):,} ({100*n_lifted/len(ec_weights_raw):.3f}%)")
#
# ec_weights_valid = ec_weights_raw[ec_weights_raw['position_hg38'].notna()].copy()
# ec_weights_valid['position_hg38'] = ec_weights_valid['position_hg38'].astype(int)
# ec_weights_valid.to_csv('/home/jupyter/prs_weights_hg38.csv', index=False)


In [ ]:
# Loading the final cohort (from EC_Cohort_Definition.ipynb) and the lifted PRS weights
r_cohort = subprocess.run(['gsutil', '-u', project_id, 'cp', f'{bucket_output}/02_ec_final_cohort_excluded.csv', '/home/jupyter/02_ec_final_cohort_excluded.csv'], capture_output=True, text=True)
if r_cohort.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_cohort.returncode}): {r_cohort.stderr}")
r_weights = subprocess.run(['gsutil', '-u', project_id, 'cp', f'{bucket_output}/prs_weights_hg38.csv', '/home/jupyter/prs_weights_hg38.csv'], capture_output=True, text=True)
if r_weights.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_weights.returncode}): {r_weights.stderr}")

final_cohort = pd.read_csv('/home/jupyter/02_ec_final_cohort_excluded.csv', low_memory=False)
print(f"Final cohort: {len(final_cohort):,} ({(final_cohort['EC_case']==1).sum():,} cases, {(final_cohort['EC_case']==0).sum():,} controls)")

prs_weights = pd.read_csv('/home/jupyter/prs_weights_hg38.csv')
prs_weights_valid = prs_weights[prs_weights['position_hg38'].notna()].copy()
prs_weights_valid['position_hg38'] = prs_weights_valid['position_hg38'].astype(int)
print(f"Lifted SNP weights: {len(prs_weights_valid):,} / {len(prs_weights):,}")


In [ ]:
bed_ranges = prs_weights_valid[['Chrom', 'position_hg38']].copy()
bed_ranges['chrom_plink'] = 'chr' + bed_ranges['Chrom'].astype(str)
bed_ranges['start'] = bed_ranges['position_hg38'] - 1
bed_ranges['end'] = bed_ranges['position_hg38']
bed_ranges = bed_ranges[['chrom_plink', 'start', 'end']].sort_values(['chrom_plink', 'start'])
bed_ranges.to_csv('/home/jupyter/prs_snp_positions.bed', sep='\t', header=False, index=False)

keep_ids = final_cohort[['person_id']].copy()
keep_ids.columns = ['#IID']
keep_ids.to_csv('/home/jupyter/cohort_person_ids.txt', sep='\t', index=False)

for local_f, remote_name in [
    ('/home/jupyter/prs_snp_positions.bed', 'prs_snp_positions.bed'),
    ('/home/jupyter/cohort_person_ids.txt', 'cohort_person_ids.txt'),
]:
    r = subprocess.run(['gsutil', '-u', project_id, 'cp', local_f, f'{bucket_scripts}/{remote_name}'], capture_output=True, text=True)
    print(remote_name, '->', 'OK' if r.returncode == 0 else r.stderr)


In [ ]:
# Staging the plink2 binary in the scripts bucket if not already there
result = subprocess.run(['gsutil', '-u', project_id, 'ls', f'{bucket_scripts}/plink2'], capture_output=True, text=True)
if result.returncode == 0:
    print("plink2 binary already staged in bucket_scripts")
else:
    subprocess.run(['wget', '-q', 'https://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip', '-O', '/home/jupyter/plink2.zip'])
    subprocess.run(['unzip', '-o', '-q', '/home/jupyter/plink2.zip', '-d', '/home/jupyter/plink2_extracted/'])
    subprocess.run(['gsutil', '-u', project_id, 'cp', '/home/jupyter/plink2_extracted/plink2', f'{bucket_scripts}/plink2'])
    print("plink2 binary staged")

In [ ]:
extract_script = '''#!/bin/bash
set -o pipefail
set -o errexit

chmod +x "${plink2_bin}"

mkdir -p /mnt/data/pfile
ln -s "${pgen_file}" /mnt/data/pfile/chr.pgen
ln -s "${psam_file}" /mnt/data/pfile/chr.psam
ln -s "${pvar_file}" /mnt/data/pfile/chr.pvar

"${plink2_bin}" \\
    --pfile /mnt/data/pfile/chr \\
    --extract bed0 "${bed_file}" \\
    --keep "${keep_file}" \\
    --threads 7 \\
    --memory 30000 \\
    --make-pgen \\
    --out "${output_folder}/prs_chr${CHR}_extracted"

echo "Done chr${CHR}"
'''

with open('/home/jupyter/dsub_PRS_extract.sh', 'w') as f:
    f.write(extract_script)

result = subprocess.run(
    ['gsutil', '-u', project_id, 'cp', '/home/jupyter/dsub_PRS_extract.sh', f'{bucket_scripts}/dsub_PRS_extract.sh'],
    capture_output=True, text=True
)
print(result.stdout, result.stderr)


In [ ]:
# Submitting one dsub job per chromosome to extract the PRS SNP positions for the cohort.
# Uncomment to rerun from scratch (~4-5 hours for all 22 chromosomes)
# Per-chromosome runtime scales with chromosome size, not cohort size, since plink2 scans the full source pgen regardless of --keep size)

# def submit_chr_extraction(chrom):
#     output_folder = f"{bucket_output}/prs_extraction/chr{chrom}"
#     logs_folder = f"{output_folder}/logs"
#     pgen_base = f"gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr{chrom}"

#     cmd = f"""dsub \\
#     --provider google-batch \\
#     --user-project {project_id} \\
#     --project {project_id} \\
#     --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/ubuntu:latest" \\
#     --network "global/networks/network" \\
#     --use-private-address \\
#     --subnetwork "regions/us-central1/subnetworks/subnetwork" \\
#     --service-account "$(gcloud config get-value account)" \\
#     --user "{DSUB_USER_NAME}" \\
#     --regions us-central1 \\
#     --logging "{logs_folder}/{{job-id}}-{{task-id}}.log" \\
#     --boot-disk-size 50 \\
#     --disk-size 130 \\
#     --machine-type n2-standard-8 \\
#     --name "prs-chr{chrom}" \\
#     --script "{bucket_scripts}/dsub_PRS_extract.sh" \\
#     --env GOOGLE_PROJECT={project_id} \\
#     --env output_folder="{output_folder}" \\
#     --env CHR={chrom} \\
#     --input plink2_bin="{bucket_scripts}/plink2" \\
#     --input pgen_file="{pgen_base}.pgen" \\
#     --input psam_file="{pgen_base}.psam" \\
#     --input pvar_file="{pgen_base}.pvar" \\
#     --input bed_file="{bucket_scripts}/prs_snp_positions.bed" \\
#     --input keep_file="{bucket_scripts}/cohort_person_ids.txt" \\
#     --output-recursive output_folder="{output_folder}"
# """
#     result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
#     print(result.stdout)
#     if result.returncode != 0:
#         print("ERROR:", result.stderr)
#     return result.stdout.strip().splitlines()[0] if result.stdout else None

# job_ids = {}
# for chrom in range(1, 23):
#     job_id = submit_chr_extraction(chrom)
#     job_ids[chrom] = job_id
#     print(f"chr{chrom}: {job_id}")

# print(f"\nSubmitted {len(job_ids)} extraction jobs")

In [ ]:
# Checking job status for above (uncomment to run)

# result = subprocess.run(
#     ['dstat', '--provider', 'google-batch', '--project', project_id,
#      '--location', 'us-central1', '--jobs'] + list(job_ids.values()) +
#     ['--users', DSUB_USER_NAME, '--status', '*'],
#     capture_output=True, text=True
# )
# print(result.stdout)

In [ ]:
import subprocess

missing = []
for chrom in range(1, 23):
    r = subprocess.run(
        ['gsutil', '-u', project_id, 'stat', f'{bucket_output}/prs_extraction/chr{chrom}/prs_chr{chrom}_extracted.pgen'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        missing.append(chrom)
        print(f"chr{chrom}: MISSING")
    else:
        # pulling the size line out of the gsutil stat output
        size_line = [l for l in r.stdout.splitlines() if 'Content-Length' in l]
        print(f"chr{chrom}: OK  {size_line[0].strip() if size_line else ''}")

print()
if missing:
    print(f"Missing chromosomes: {missing} — extraction is incomplete, would need to (re)run these.")
else:
    print("All 22 chromosomes present. Extraction does NOT need to be re-run.")

## Allele harmonisation


In [ ]:
# Loading every extracted chromosome's real REF/ALT alleles
pvar_all = []
failed_chroms = []
for chrom in range(1, 23):
    local_path = f'/home/jupyter/chr{chrom}_extracted.pvar'
    r = subprocess.run(['gsutil', '-u', project_id, 'cp', f'{bucket_output}/prs_extraction/chr{chrom}/prs_chr{chrom}_extracted.pvar', local_path], capture_output=True, text=True)
    if r.returncode != 0:
        failed_chroms.append(chrom)
        print(f"chr{chrom}: gsutil cp FAILED (exit {r.returncode}): {r.stderr}")
        continue

    with open(local_path) as f:
        header_line = None
        for i, line in enumerate(f):
            if not line.startswith('##'):
                header_line = i
                break

    df = pd.read_csv(local_path, sep='\t', skiprows=header_line, header=0)
    df.columns = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'INFO']
    df['chrom_num'] = chrom
    pvar_all.append(df[['chrom_num', 'POS', 'ID', 'REF', 'ALT']])
    print(f"chr{chrom}: {len(df):,} variants loaded")

if failed_chroms:
    raise RuntimeError(f"gsutil cp failed to download .pvar for chromosome(s) {failed_chroms} -- "
                       f"cannot build a complete pvar_combined from a partial download.")

pvar_combined = pd.concat(pvar_all, ignore_index=True)
print(f"\nTotal variants across all extracted chromosomes: {len(pvar_combined):,}")


In [ ]:
# Before deduplicating by position, checking how many PRS positions have more than one .pvar record
# The next cell keeps only the first record per position, which could discard the correct match if that record isn't the one kept for a given weight.
pvar_lookup_pre = pvar_combined.copy()
pvar_lookup_pre['plink_id_candidate'] = 'chr' + pvar_lookup_pre['chrom_num'].astype(str) + ':' + pvar_lookup_pre['POS'].astype(str)

position_counts = pvar_lookup_pre['plink_id_candidate'].value_counts()
multi_record_positions = position_counts[position_counts > 1]

print(f"Total extracted .pvar records: {len(pvar_lookup_pre):,}")
print(f"Unique chromosome:position combinations: {pvar_lookup_pre['plink_id_candidate'].nunique():,}")
print(f"Positions with more than one .pvar record: {len(multi_record_positions):,}")

weight_positions = set(prs_weights_valid['plink_id_candidate']) if 'plink_id_candidate' in prs_weights_valid.columns else \
    set('chr' + prs_weights_valid['Chrom'].astype(str) + ':' + prs_weights_valid['position_hg38'].astype(str))
multi_record_affecting_weights = weight_positions & set(multi_record_positions.index)
print(f"Weighted SNP positions affected by multi-record ambiguity: {len(multi_record_affecting_weights):,} "
      f"/ {len(weight_positions):,} ({100*len(multi_record_affecting_weights)/len(weight_positions):.3f}%)")

In [ ]:
# For each weight, finding its real REF/ALT and determine the correct effect-allele match (direct or swapped)
prs_weights_valid['plink_id_candidate'] = 'chr' + prs_weights_valid['Chrom'].astype(str) + ':' + prs_weights_valid['position_hg38'].astype(str)

pvar_lookup = pvar_combined.copy()
pvar_lookup['plink_id_candidate'] = 'chr' + pvar_lookup['chrom_num'].astype(str) + ':' + pvar_lookup['POS'].astype(str)
pvar_lookup = pvar_lookup[['plink_id_candidate', 'ID', 'REF', 'ALT']].drop_duplicates(subset='plink_id_candidate')

merged = prs_weights_valid.merge(pvar_lookup, on='plink_id_candidate', how='left')
print(f"Weights matched to a real pvar position: {merged['REF'].notna().sum():,} / {len(merged):,}")

def determine_score_row(row):
    if pd.isna(row['REF']):
        return None, None, None, 'position_unavailable'
    alts = str(row['ALT']).split(',')
    if row['A1'] == row['REF'] and row['A2'] in alts:
        return row['ID'], row['A1'], row['A1Effect'], 'direct'
    elif row['A1'] in alts and row['A2'] == row['REF']:
        return row['ID'], row['A1'], row['A1Effect'], 'swapped'
    else:
        return None, None, None, 'allele_pair_incompatible'

results = merged.apply(lambda r: pd.Series(determine_score_row(r)), axis=1)
results.columns = ['MATCHED_VARIANT_ID', 'SCORE_A1', 'SCORE_WEIGHT', 'match_type']
merged = pd.concat([merged.drop(columns=[c for c in ['MATCHED_VARIANT_ID','SCORE_A1','SCORE_WEIGHT','match_type'] if c in merged.columns]), results], axis=1)

# match_type is derived directly from determine_score_row's own branching, so it exactly reflects why each row did or didn't get a MATCHED_VARIANT_ID
assert (merged['MATCHED_VARIANT_ID'].notna() == merged['match_type'].isin(['direct', 'swapped'])).all(), (
    "match_type no longer agrees with which rows actually got a MATCHED_VARIANT_ID"
)
print(merged['match_type'].value_counts())

In [ ]:
# Building and saving the corrected v2 score file
score_file_v2 = merged[merged['MATCHED_VARIANT_ID'].notna()][['MATCHED_VARIANT_ID', 'SCORE_A1', 'SCORE_WEIGHT']].copy()
score_file_v2.columns = ['ID', 'A1', 'WEIGHT']
print(f"Final corrected score file: {len(score_file_v2):,} rows")

score_file_v2.to_csv('/home/jupyter/prs_score_file_v2.txt', sep='\t', index=False)
r_score_v2 = subprocess.run(['gsutil', '-u', project_id, 'cp', '/home/jupyter/prs_score_file_v2.txt', f'{bucket_scripts}/prs_score_file_v2.txt'], capture_output=True, text=True)
if r_score_v2.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_score_v2.returncode}): {r_score_v2.stderr}")


In [ ]:
# Known-SNP check 1: rs4430796 (HNF1B), direct match (confirming the merged/matched row looks correct)
check = merged[merged['Name'] == 'rs4430796']
print(check[['Name', 'A1', 'A2', 'REF', 'ALT', 'MATCHED_VARIANT_ID', 'SCORE_A1', 'SCORE_WEIGHT', 'match_type']])

# Known-SNP check 2: rs11841589 (KLF5), swapped match (T is the known protective allele)
check2 = merged[merged['Name'] == 'rs11841589']
print(check2[['Name', 'A1', 'A2', 'REF', 'ALT', 'MATCHED_VARIANT_ID', 'SCORE_A1', 'SCORE_WEIGHT', 'match_type']])

In [ ]:
# Allele-frequency agreement (parsing AF from each extracted .pvar's INFO field, comparing to weights file)
def parse_af(info_str):
    for field in str(info_str).split(';'):
        if field.startswith('AF='):
            return field[3:].split(',')
    return None

pvar_af = []
for chrom in range(1, 23):
    local_path = f'/home/jupyter/chr{chrom}_extracted.pvar'
    with open(local_path) as f:
        for line in f:
            if line.startswith('##') or line.startswith('#CHROM'):
                continue
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 6:
                continue
            chrom_p, pos, vid, ref, alt, info = parts[0], parts[1], parts[2], parts[3], parts[4], parts[5]
            afs = parse_af(info)
            pvar_af.append({'chrom_num': chrom, 'POS': int(pos), 'ID': vid, 'REF': ref, 'ALT': alt, 'ALT_AF': afs[0] if afs else None})

pvar_af_df = pd.DataFrame(pvar_af)
pvar_af_df['ALT_AF'] = pd.to_numeric(pvar_af_df['ALT_AF'], errors='coerce')

af_check = merged[merged['MATCHED_VARIANT_ID'].notna()].copy()
af_check = af_check.merge(pvar_af_df[['ID', 'ALT_AF']].rename(columns={'ID': 'MATCHED_VARIANT_ID'}), on='MATCHED_VARIANT_ID', how='left')
af_check['aou_A1_freq'] = np.where(af_check['A1'] == af_check['REF'], 1 - af_check['ALT_AF'], af_check['ALT_AF'])
af_check['freq_diff'] = (af_check['A1Frq'] - af_check['aou_A1_freq']).abs()
af_check['is_palindromic'] = af_check.apply(lambda r: {r['A1'], r['A2']} in [{'A','T'}, {'C','G'}], axis=1)

print(f"Palindromic SNPs in score file: {af_check['is_palindromic'].sum():,} (expect 0 -- pre-removed by Tracy)")
print(f"Frequency broadly agrees (|diff| < 0.10): {100*(af_check['freq_diff']<0.10).mean():.1f}% of matched SNPs")


## PRS scoring


In [ ]:
score_script = '''#!/bin/bash
set -o pipefail
set -o errexit

chmod +x "${plink2_bin}"

mkdir -p /mnt/data/pfile
ln -s "${pgen_file}" /mnt/data/pfile/chr.pgen
ln -s "${psam_file}" /mnt/data/pfile/chr.psam
ln -s "${pvar_file}" /mnt/data/pfile/chr.pvar

"${plink2_bin}" \\
    --pfile /mnt/data/pfile/chr \\
    --score "${score_file}" 1 2 3 header no-mean-imputation list-variants \\
    --threads 7 \\
    --memory 30000 \\
    --out "${output_folder}/prs_chr${CHR}_score"

echo "Done chr${CHR}"
'''

with open('/home/jupyter/dsub_PRS_score.sh', 'w') as f:
    f.write(score_script)
r_score_script = subprocess.run(['gsutil', '-u', project_id, 'cp', '/home/jupyter/dsub_PRS_score.sh', f'{bucket_scripts}/dsub_PRS_score.sh'], capture_output=True, text=True)
if r_score_script.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_score_script.returncode}): {r_score_script.stderr}")


In [ ]:
# Submitting one dsub job per chromosome to score the extracted genotypes against prs_score_file_v2.txt
# Uncomment to rerun

# def submit_chr_scoring(chrom, score_file_name='prs_score_file_v2.txt', tag='scv2'):
#     output_folder = f"{bucket_output}/prs_scoring_v2/chr{chrom}"
#     logs_folder = f"{output_folder}/logs"
#     extracted_base = f"{bucket_output}/prs_extraction/chr{chrom}/prs_chr{chrom}_extracted"

#     cmd = f"""dsub \\
#     --provider google-batch \\
#     --user-project {project_id} \\
#     --project {project_id} \\
#     --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/ubuntu:latest" \\
#     --network "global/networks/network" \\
#     --use-private-address \\
#     --subnetwork "regions/us-central1/subnetworks/subnetwork" \\
#     --service-account "$(gcloud config get-value account)" \\
#     --user "{DSUB_USER_NAME}" \\
#     --regions us-central1 \\
#     --logging "{logs_folder}/{{job-id}}-{{task-id}}.log" \\
#     --boot-disk-size 30 \\
#     --disk-size 30 \\
#     --machine-type n2-standard-4 \\
#     --name "{tag}-chr{chrom}" \\
#     --script "{bucket_scripts}/dsub_PRS_score.sh" \\
#     --env GOOGLE_PROJECT={project_id} \\
#     --env output_folder="{output_folder}" \\
#     --env CHR={chrom} \\
#     --input plink2_bin="{bucket_scripts}/plink2" \\
#     --input pgen_file="{extracted_base}.pgen" \\
#     --input psam_file="{extracted_base}.psam" \\
#     --input pvar_file="{extracted_base}.pvar" \\
#     --input score_file="{bucket_scripts}/{score_file_name}" \\
#     --output-recursive output_folder="{output_folder}"
# """
#     result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
#     print(result.stdout)
#     if result.returncode != 0:
#         print("ERROR:", result.stderr)
#     return result.stdout.strip().splitlines()[0] if result.stdout else None

# score_job_ids = {}
# for chrom in range(1, 23):
#     job_id = submit_chr_scoring(chrom)
#     score_job_ids[chrom] = job_id
#     print(f"chr{chrom}: {job_id}")


In [ ]:
# Checking job status from above (uncomment to rerun)

# result = subprocess.run(
#     ['dstat', '--provider', 'google-batch', '--project', project_id,
#      '--location', 'us-central1', '--jobs'] + list(score_job_ids.values()) +
#     ['--users', DSUB_USER_NAME, '--status', '*'],
#     capture_output=True, text=True
# )
# print(result.stdout)

In [ ]:
# Checking whether the already-saved prs_scores_final_complete.csv already covers the full current cohort 
# Doing this before re-downloading and re-combining 22 chromosomes from prs_scoring_v2 as that would be an expensive, unnecessary step if nothing has changed
r_check = subprocess.run(['gsutil', '-u', project_id, 'cp', f'{bucket_output}/prs_scores_final_complete.csv', '/home/jupyter/_saved_check.csv'], capture_output=True, text=True)
if r_check.returncode == 0:
    _saved = pd.read_csv('/home/jupyter/_saved_check.csv', low_memory=False, usecols=['#IID', 'EC_case'])
    saved_is_complete = (
        set(final_cohort['person_id']) == set(_saved['#IID'])
        and int(_saved['EC_case'].sum()) == int(final_cohort['EC_case'].sum())
    )
else:
    saved_is_complete = False

if saved_is_complete:
    print("Already saved and complete: prs_scores_final_complete.csv covers the full current "
          f"cohort (N={len(_saved):,}). Skipping the recombine below.")
else:
    print("Saved file is missing or stale relative to the current cohort -- recombining from "
          "prs_scoring_v2/ below.")


In [ ]:
# Summing each chromosome's per-chromosome PLINK2 score contribution into a genome-wide raw PR
# Standardising this against the control mean/SD
# Skipped entirely if the check above already confirmed the saved file is complete
# ana_fixed is loaded from that file instead

if saved_is_complete:
    ana_fixed = _saved.merge(final_cohort.rename(columns={'person_id': '#IID'}), on='#IID', how='left', suffixes=('', '_dup'))
    ana_fixed = ana_fixed[[c for c in ana_fixed.columns if not c.endswith('_dup')]]
    ctrl_m = ana_fixed.loc[ana_fixed['EC_case'] == 0, 'PRS_raw'].mean() if 'PRS_raw' in ana_fixed.columns else None
    print(f"Loaded ana_fixed from the already-complete saved file (N={len(ana_fixed):,}).")
else:
    from functools import reduce
    import statsmodels.api as sm

    all_scores = []
    failed_score_chroms = []
    for chrom in range(1, 23):
        local_path = f'/home/jupyter/chr{chrom}_score.sscore'
        r = subprocess.run(['gsutil', '-u', project_id, 'cp', f'{bucket_output}/prs_scoring_v2/chr{chrom}/prs_chr{chrom}_score.sscore', local_path], capture_output=True, text=True)
        if r.returncode != 0:
            failed_score_chroms.append(chrom)
            print(f"chr{chrom}: gsutil cp FAILED (exit {r.returncode}): {r.stderr}")
            continue
        df = pd.read_csv(local_path, sep='\t')
        df['chr_sum'] = df['SCORE1_AVG'] * df['ALLELE_CT']
        df = df[['#IID', 'chr_sum', 'ALLELE_CT']].rename(columns={'chr_sum': f'sum_chr{chrom}', 'ALLELE_CT': f'ct_chr{chrom}'})
        all_scores.append(df)

    if failed_score_chroms:
        print(f"WARNING: gsutil cp failed to download .sscore for chromosome(s) {failed_score_chroms} -- "
              f"this will surface below as incomplete allele counts for every person (ct_chr* check).")

    final_scores_refresh = reduce(lambda left, right: left.merge(right, on='#IID', how='outer'), all_scores)
    sum_cols = [c for c in final_scores_refresh.columns if c.startswith('sum_chr')]
    final_scores_refresh['PRS_raw'] = final_scores_refresh[sum_cols].sum(axis=1)

    final_cohort_ids = final_cohort[['person_id']].rename(columns={'person_id':'#IID'})
    ana_fixed = final_scores_refresh.merge(final_cohort_ids, on='#IID', how='inner').merge(
        final_cohort.rename(columns={'person_id':'#IID'}), on='#IID', how='left'
    )

    ctrl_m = ana_fixed.loc[ana_fixed['EC_case']==0, 'PRS_raw'].mean()
    ctrl_s = ana_fixed.loc[ana_fixed['EC_case']==0, 'PRS_raw'].std()
    ana_fixed['PRS_z'] = (ana_fixed['PRS_raw'] - ctrl_m) / ctrl_s

    # Verifying every chromosome actually contributed a real allele count for every person before saving
    ct_check_cols = [c for c in ana_fixed.columns if c.startswith('ct_chr')]
    n_incomplete = ana_fixed[ct_check_cols].isna().any(axis=1).sum()
    print(f"Incomplete people: {n_incomplete} (must be 0 before saving)")

    # Cohort-coverage check (the INNER join above would silently shrink ana_fixed, not error if prs_scoring_v2/ is stale relative to final_cohort)
    missing_from_scores = set(final_cohort['person_id']) - set(ana_fixed['#IID'])
    cohort_is_complete = (
        len(missing_from_scores) == 0
        and len(ana_fixed) == len(final_cohort)
        and int(ana_fixed['EC_case'].sum()) == int(final_cohort['EC_case'].sum())
    )

    if not cohort_is_complete:
        print("WARNING: ana_fixed does not cover the full current cohort.")
        print(f"  {len(missing_from_scores):,} cohort member(s) are missing from the PRS scores "
              f"(prs_scoring_v2/ is stale relative to the current final_cohort).")
        print(f"  ana_fixed: N={len(ana_fixed):,}, cases={int(ana_fixed['EC_case'].sum()):,} "
              f"| final_cohort: N={len(final_cohort):,}, cases={int(final_cohort['EC_case'].sum()):,}")
        print("  To fix: regenerate cohort_person_ids.txt from the current final_cohort and "
              "re-run the full extraction/scoring cells above.")
    else:
        print(f"Confirmed: ana_fixed exactly matches the current cohort "
              f"(N={len(ana_fixed):,}, cases={int(ana_fixed['EC_case'].sum()):,})")

        pc_cols_check = sorted([c for c in ana_fixed.columns if c.startswith('PC') and c[2:].isdigit()], key=lambda x: int(x[2:]))[:10]
        model_check = ana_fixed.dropna(subset=['PRS_z'] + pc_cols_check)
        X_check = sm.add_constant(model_check[['PRS_z'] + pc_cols_check])
        y_check = model_check['EC_case']
        logit_check = sm.Logit(y_check, X_check).fit(disp=0)
        print(f"OR: {np.exp(logit_check.params['PRS_z']):.4f}  (must be ~1.70, NOT ~5.22)")


In [ ]:
# Only overwriting the saved score file if a fresh recombine happened AND it was confirmed complete
# Otherwise this would silently overwrite a known-good, already-complete prs_scores_final_complete.csv with a stale/incomplete one. 
# Nothing to do if the earlier check already confirmed the saved file is complete (saved_is_complete), since it was never touched in that case
if not saved_is_complete and cohort_is_complete:
    ana_fixed.to_csv('/home/jupyter/prs_scores_final_complete.csv', index=False)
    r = subprocess.run(['gsutil','-u',project_id,'cp','/home/jupyter/prs_scores_final_complete.csv',
        f'{bucket_output}/prs_scores_final_complete.csv'], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"gsutil cp failed (exit {r.returncode}): {r.stderr}")
    print(f"Saved: N={len(ana_fixed):,}, cases={int((ana_fixed['EC_case']==1).sum()):,}, controls={int((ana_fixed['EC_case']==0).sum()):,}")
elif saved_is_complete:
    print("Nothing to save: the already-saved file is complete and was never touched.")
else:
    print("Skipped save: ana_fixed is incomplete relative to the current cohort (see warning above).")
    print("Not overwriting prs_scores_final_complete.csv until the cohort gap is resolved.")


In [ ]:
# SNP-coverage accounting (checking exactly how many of the ~1.14M weighted SNPs contributed to the final PRS)
total_used = 0
failed_vars_chroms = []
for chrom in range(1, 23):
    lp = f'/home/jupyter/chr{chrom}_score.sscore.vars'
    r = subprocess.run(['gsutil','-u',project_id,'cp', f'{bucket_output}/prs_scoring_v2/chr{chrom}/prs_chr{chrom}_score.sscore.vars', lp], capture_output=True, text=True)
    if r.returncode != 0:
        failed_vars_chroms.append(chrom)
        print(f"chr{chrom}: gsutil cp FAILED (exit {r.returncode}): {r.stderr}")
        continue
    with open(lp) as f:
        total_used += sum(1 for _ in f)

if failed_vars_chroms:
    raise RuntimeError(f"gsutil cp failed to download .sscore.vars for chromosome(s) {failed_vars_chroms} -- "
                       f"total_used would silently undercount SNP coverage.")

print(f"Original weights file: {len(prs_weights):,}")
print(f"  ...with a valid lifted hg38 position: {len(prs_weights_valid):,} ({100*len(prs_weights_valid)/len(prs_weights):.1f}%)")
print(f"  ...matched to a real AoU extracted variant: {merged['MATCHED_VARIANT_ID'].notna().sum():,}")
print(f"  ...actually used by plink2 in final scoring: {total_used:,}")
print(f"\nNet SNP coverage: {100*total_used/len(prs_weights):.2f}% of the original weighted SNPs contribute to the final PRS")

## Coverage checks


In [ ]:
# Reloading the freshly-saved file directly from the bucket (not from kernel memory) to verify what's actually persisted
r_verify_final = subprocess.run(['gsutil','-u',project_id,'cp', f'{bucket_output}/prs_scores_final_complete.csv', '/home/jupyter/verify_final.csv'], capture_output=True, text=True)
if r_verify_final.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_verify_final.returncode}): {r_verify_final.stderr}")
verify = pd.read_csv('/home/jupyter/verify_final.csv')

print("Rows:", len(verify))
print("Cases:", int((verify['EC_case']==1).sum()))
print("Controls:", int((verify['EC_case']==0).sum()))
print("Duplicate IIDs:", verify['#IID'].duplicated().sum())
print("Missing PRS_z:", verify['PRS_z'].isna().sum())
print("Missing PRS_raw:", verify['PRS_raw'].isna().sum())

# Cross-checking against the current cohort definition
r_verify_cohort = subprocess.run(['gsutil','-u',project_id,'cp', f'{bucket_output}/02_ec_final_cohort_excluded.csv', '/home/jupyter/verify_cohort.csv'], capture_output=True, text=True)
if r_verify_cohort.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {r_verify_cohort.returncode}): {r_verify_cohort.stderr}")
current = pd.read_csv('/home/jupyter/verify_cohort.csv', low_memory=False)
missing = set(current['person_id']) - set(verify['#IID'])
print("\nCurrent cohort N:", len(current), "| cases:", int((current['EC_case']==1).sum()))
print("People in current cohort missing from PRS scores:", len(missing))

In [ ]:
# Targeted integrity checks on the saved file 
# Checking case labels match the cohort file, 
# Checking every chromosome has a positive (not just non-null) contribution
# Checking PRS_raw/PRS_z reconstruct correctly from their own stored inputs)
label_check = verify[['#IID', 'EC_case']].merge(
    current[['person_id', 'EC_case']],
    left_on='#IID', right_on='person_id',
    how='outer', validate='one_to_one', indicator=True, suffixes=('_prs', '_cohort')
)
assert label_check['_merge'].eq('both').all(), "verify and current cohort do not have identical participant sets"
assert label_check['EC_case_prs'].eq(label_check['EC_case_cohort']).all(), "EC_case label mismatch between saved PRS file and current cohort file"
print("Case-label membership check: passed (all participants present in both files, all EC_case labels agree)")

sum_cols = [f'sum_chr{i}' for i in range(1, 23)]
ct_cols = [f'ct_chr{i}' for i in range(1, 23)]
assert verify[sum_cols + ct_cols].notna().all().all(), "one or more chromosome sum/count columns contain nulls"
assert verify[ct_cols].gt(0).all().all(), "one or more participants have a zero allele count on a chromosome"
print("Chromosome contribution check: passed (all 22 sum_chr/ct_chr columns present, non-null, and ct_chr > 0 for every participant)")

import numpy as np
np.testing.assert_allclose(
    verify['PRS_raw'], verify[sum_cols].sum(axis=1), rtol=1e-8, atol=1e-8,
    err_msg="PRS_raw does not equal the sum of its 22 chromosome contributions"
)
print("PRS_raw reconstruction check: passed (PRS_raw == sum of sum_chr1..sum_chr22 for every participant)")

controls_raw = verify.loc[verify['EC_case'] == 0, 'PRS_raw']
expected_z = (verify['PRS_raw'] - controls_raw.mean()) / controls_raw.std()
np.testing.assert_allclose(
    verify['PRS_z'], expected_z, rtol=1e-8, atol=1e-8,
    err_msg="PRS_z does not match standardisation against the current file's own control mean/SD"
)
print("PRS_z standardisation check: passed (PRS_z matches (PRS_raw - control_mean) / control_std, recomputed from verify itself)")


In [ ]:
# Checking genotype-call coverage per person (--score ...no-mean-imputation silently drops missing genotypes rather than imputing them)
# Runs on `verify` (the saved, reloaded-fresh file), not ana_fixed.
ct_cols_total = [c for c in verify.columns if c.startswith('ct_chr')]
verify['total_allele_ct'] = verify[ct_cols_total].sum(axis=1)

print("Distribution of total observed allele count (sum across 22 chromosomes), overall:")
print(verify['total_allele_ct'].describe())

if 'ancestry_pred' in verify.columns:
    print("\nBy ancestry group:")
    print(verify.groupby('ancestry_pred')['total_allele_ct'].describe()[['count','mean','std','min','max']])

print("\nBy case status:")
print(verify.groupby('EC_case')['total_allele_ct'].describe()[['count','mean','std','min','max']])

# A material correlation here would point to differential genotype missingness, rather than
# the population-genetic explanations discussed in Section 4.4, as a contributor to the
# ancestry-dependent PRS scale differences reported in the dissertation.
corr_with_prs = verify[['total_allele_ct', 'PRS_raw']].corr().iloc[0,1]
print(f"\nCorrelation between total_allele_ct and PRS_raw: {corr_with_prs:.4f}")
